# Event Segmentation Plots

This script loads in the event segmentation data, organizes it into a DataFrame, and generates:  
1. Raster plots ("spikes" corresponding to  participant marks)
2. KDE plots representing the "spike" distributions
3. A summary output containing the # raters and # events per walk

> Author:    Justin Campbell    
> Contact:   justin.campbell@hsc.utah.edu     
> Version:   04-14-2022   

Modified on 10/10/2025 to fix the issue with seaborn KDE scaled values by Alireza Kazemi

## 1. Import Libraries

In [1]:
import os
import glob
import datetime
import statistics
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity

Functions


In [4]:
def ComputeHISTandKDE(walkDF, BW = 2, KDERes = 100):
    numVoters = walkDF['pID'].nunique()
    totalVotes = len(walkDF['Event'])
    tMax = math.ceil(walkDF['Event'].max())

    # Histogram (counts per voter, binwidth=BW, limits [0, tMax])
    bins = np.arange(0, tMax + BW, BW)
    histVals, binEdges = np.histogram(walkDF['Event'], bins=bins)
    binCenters = binEdges[:-1] + np.diff(binEdges) / 2
    histVals = histVals / numVoters

    # KDE (Gaussian, bandwidth=BW) evaluated on xKDE = 0:1/f:tMax
    
    xKDE = np.arange(0, tMax + 1/KDERes, 1/KDERes).reshape(-1, 1)
    x = walkDF['Event'].to_numpy().reshape(-1, 1)
    kde = KernelDensity(kernel="gaussian", bandwidth=BW).fit(x)
    kdeVals = np.exp(kde.score_samples(xKDE)) * BW * totalVotes / numVoters  # scale to match histogram

    return histVals, binCenters, kdeVals, xKDE,numVoters

In [ ]:
def genRasterHist_Fixed(ESTable, pID, walk , save = False, savepath = "",BW = 2, KDERes = 100, Only20Raters = True):
  
  # Filter data
  df = ESTable[ESTable['RWN'] == int(pID[-1])].copy()
  walkDF = df[df['Walk'] == walk].copy().reset_index(drop = True)

  # Randomly sample 20 raters
  if Only20Raters == True:
    np.random.seed(1)
    raterIDs = walkDF['pIDNum'].unique()
    keepRaters = np.random.choice(raterIDs, size = 20, replace = False)
    walkDF = walkDF[walkDF['pIDNum'].isin(keepRaters)]
  
  tMax = np.ceil(walkDF['Event'].max() / 10) * 10

  # Dicts for summary data
  nRaters = {}

  # Raster & KDE
  if not walkDF.empty:
    rasterEvents = []
    for ii in walkDF['pID'].unique():
      nRaters = len(walkDF['pID'].unique())
      pWalkDF = walkDF[walkDF['pID'] == ii]
      rasterEvents.append(pWalkDF['Event'].values)
    # Compute histogram and KDE (new)
    fig, axes, hist = plot_ES_raster_KDE(rasterEvents, walkDF, tMax, BW, KDERes, pID, walk, nRaters)
    # Compute histogram and KDE (new)
    histVals, binCenters, kdeVals, xKDE, numVoters = ComputeHISTandKDE(
        walkDF, BW=BW, KDERes=KDERes
    )
    fig.show()


    

    if save == True:
      # Figure
      save_str = pID + '_' + str(walk) + '_ESPlots.pdf'
      fig.savefig(os.path.join(savepath, 'Figures', save_str), dpi = 600, bbox_inches = 'tight')
      # Export raw KDE vals
      KDE_X, KDE_Y = hist.get_lines()[0].get_data()
      KDE_DF = pd.DataFrame({'X': KDE_X, 'Y': KDE_Y})
      save_str = pID + '_' + str(walk) + '_KDE.csv'
      KDE_DF.to_csv(os.path.join(savepath, 'KDEs_Old', save_str))
      # Export new KDE vals
      KDE_DF = pd.DataFrame({'X': xKDE.ravel(), 'Y': kdeVals.ravel()})
      save_str = pID + '_' + str(walk) + '_KDE.csv'
      KDE_DF.to_csv(os.path.join(savepath, 'KDEs_New', save_str))
      # Export new Hist vals
      KDE_DF = pd.DataFrame({'X': binCenters, 'Y': histVals})
      save_str = pID + '_' + str(walk) + '_Hist.csv'
      KDE_DF.to_csv(os.path.join(savepath, 'Hists_New', save_str))
    plt.close()


Load EventSegTable.csv

In [10]:
from sklearn.neighbors import KernelDensity
datapath = "Z:\Data\RealWorldNavigationCory\KDEs"
ESTable = pd.read_csv(os.path.join(datapath, 'EventSegTable.csv'))
print(ESTable.head())


   Unnamed: 0    pID  pIDNum     Video  RWN  Walk      Event
0           0  ES335     335  video5_6    5     6  16.570780
1           1  ES335     335  video5_6    5     6  45.244843
2           2  ES335     335  video5_6    5     6  57.480866
3           3  ES335     335  video5_6    5     6  70.232940
4           4  ES335     335  video5_6    5     6  85.495896


In [16]:
savepath = "D:\CAPTURE Project\GUI\EventKDE"
for i in ESTable['RWN'].unique():
  RWNTable = ESTable[ESTable['RWN'] == i]
  print('Processing RWN %s...' %i)
  for ii in RWNTable['Walk'].unique():
    genRasterHist_Fixed(
      ESTable,pID = ('RWN' + str(i)), walk = ii, 
      save = True, savepath = savepath, 
      BW = 2, KDERes = 100, Only20Raters = True)

Processing RWN 5...


C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: Us

Processing RWN 4...


C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: Us

Processing RWN 1...


C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: Us

Processing RWN 2...


C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: Us

Processing RWN 3...


C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
C:\Users\alire\AppData\Local\Temp\ipykernel_26728\4140492491.py:32: Us

Include all Raters

In [14]:
savepath = "D:\CAPTURE Project\GUI\EventKDE"
for i in ESTable['RWN'].unique():
  RWNTable = ESTable[ESTable['RWN'] == i]
  print('Processing RWN %s...' %i)
  for ii in RWNTable['Walk'].unique():
    genRasterHist_Fixed(ESTable,pID = ('RWN' + str(i)), walk = ii, save = True, savepath = savepath, BW = 2, KDERes = 100, Only20Raters = False)

Processing RWN 5...
Processing RWN 4...
Processing RWN 1...
Processing RWN 2...
Processing RWN 3...


In [16]:
savepath = "D:\CAPTURE Project\GUI\EventKDE"
for i in ESTable['RWN'].unique():
  RWNTable = ESTable[ESTable['RWN'] == i]
  print('Processing RWN %s...' %i)
  for ii in RWNTable['Walk'].unique():
    genRasterHist_Fixed(ESTable,pID = ('RWN' + str(i)), walk = ii, save = True, savepath = savepath, BW = 4, KDERes = 100, Only20Raters = False)

Processing RWN 5...
Processing RWN 4...
Processing RWN 1...
Processing RWN 2...
Processing RWN 3...


Debug genRasterHist single walk and participant

In [ ]:
pID = 'RWN3'
walk = 5
savepath = "D:\CAPTURE Project\GUI\EventKDE"

In [ ]:

# Initialize figure
fig = plt.figure(figsize = (30,10))

# Define grid spec
height_ratios = [1,1]
gs = gridspec.GridSpec(ncols=1, nrows=2, wspace = 0.25, figure=fig, height_ratios = height_ratios)
axA = fig.add_subplot(gs[0,0]) # Raster
axB = fig.add_subplot(gs[1,0], sharex = axA) # KDE

# Filter data
df = ESTable[ESTable['RWN'] == int(pID[-1])].copy()
walkDF = df[df['Walk'] == walk].copy().reset_index(drop = True)

# walkDF.to_csv(os.path.join(savepath, 'KDEs', 'WalDF.csv'))

print(walkDF["pID"].nunique())

# Randomly sample 20 raters
np.random.seed(1)
raterIDs = walkDF['pIDNum'].unique()
keepRaters = np.random.choice(raterIDs, size = 20, replace = False)
walkDF = walkDF[walkDF['pIDNum'].isin(keepRaters)]
tMax = np.ceil(walkDF['Event'].max() / 10) * 10
print(tMax)
# Dicts for summary data
# nRaters = {}
# nEvents = {}

# Raster & KDE
BW = 2
# rasterEvents = []
# for ii in walkDF['pID'].unique():
#     nRaters = len(walkDF['pID'].unique())
#     pWalkDF = walkDF[walkDF['pID'] == ii]
#     rasterEvents.append(pWalkDF['Event'].values)
# axA.eventplot(rasterEvents, linelengths = 0.75, linewidths = 2, lineoffsets = 1, colors = sns.color_palette('crest', len(rasterEvents)))
hist1 = sns.histplot(data = walkDF, x = 'Event', binrange = [0,tMax], 
                    binwidth = BW, color = histColor,stat = 'count', ax =axA,
                    kde = True, kde_kws= {'bw_adjust': 0.05})
# hist1 = sns.histplot(data = walkDF, x = 'Event', binrange = [0,tMax], 
#                     binwidth = BW, color = histColor,stat = 'density', ax =axB,
#                     kde = True, kde_kws= {'bw_adjust': 0.05})
# hist1 = sns.kdeplot(data = walkDF, x = 'Event',bw_adjust= 0.05, ax =axB)

x = walkDF["Event"].to_numpy().reshape(-1, 1)
votersNum = walkDF["pIDNum"].nunique()
totalVotes = walkDF.shape[0]
print(totalVotes, votersNum)
kde = KernelDensity(kernel="gaussian", bandwidth=BW).fit(x)
grid = np.arange(start = x.min(), stop = x.max(), step = 1/100)[:,None]
log_dens = kde.score_samples(grid)
kdeVals = np.exp(log_dens)*BW

# Histogram (normalized to density)
hist2 = sns.histplot(data = walkDF, x = 'Event', binrange = [0,tMax], 
                    binwidth = BW, color = histColor,stat = 'count', ax =axB,
                    kde = False)

# axB.hist(x.ravel(), , density=True, alpha=0.5, color="gray", edgecolor="black")

# KDE curve
axB.plot(grid.ravel(), kdeVals, color="red", linewidth=2)

# Export raw KDE vals
KDE_X, KDE_Y = hist1.get_lines()[0].get_data()
KDE_DF = pd.DataFrame({'X': KDE_X, 'Y': KDE_Y})
save_str = pID + '_' + str(walk) + '_KDE.csv'
KDE_DF.to_csv(os.path.join(savepath, 'KDEs', save_str))

# KDE_X, KDE_Y = hist1.get_lines()[0].get_data()
# KDE_DF = pd.DataFrame({'X': KDE_X, 'Y': KDE_Y})
# save_str = pID + '_' + str(walk) + '_KDE1.csv'
# KDE_DF.to_csv(os.path.join(savepath, 'KDEs', save_str))

In [ ]:
histVals, binCenters, kdeVals, xKDE = ComputeHISTandKDE(walkDF, BW = 2, KDERes = 100)

# Plot
fig = plt.figure(figsize = (30,10))
height_ratios = [1,1]
gs = gridspec.GridSpec(ncols=1, nrows=2, wspace = 0.25, figure=fig, height_ratios = height_ratios)
ax = fig.add_subplot(gs[0,0]) 

ax.stem(binCenters, histVal)
ax.plot(xKDE.ravel(), C, '-r', linewidth=2)
ax.set_xlim(0, tMax)
ax.set_xlabel('Event')
ax.set_ylabel('Count per voter')
plt.show()


In [ ]:
# # Summary for one patient:
DF =genRasterHist(pID = 'RWN3', walk = 5, save = True, savepath = "D:\GithubRep\CAPTURE\Analysis\EventKDE")


In [ ]:


# Initialize figure
fig = plt.figure(figsize = (30,10))

# Define grid spec
height_ratios = [1,1]
gs = gridspec.GridSpec(ncols=1, nrows=2, wspace = 0.25, figure=fig, height_ratios = height_ratios)
axA = fig.add_subplot(gs[0,0]) # Raster
axB = fig.add_subplot(gs[1,0], sharex = axA) # KDE

# Filter data
df = ESTable[ESTable['RWN'] == int(pID[-1])].copy()
walkDF = df[df['Walk'] == walk].copy().reset_index(drop = True)

# walkDF.to_csv(os.path.join(savepath, 'KDEs', 'WalDF.csv'))

print(walkDF["pID"].nunique())

# Randomly sample 20 raters
np.random.seed(1)
raterIDs = walkDF['pIDNum'].unique()
keepRaters = np.random.choice(raterIDs, size = 20, replace = False)
walkDF = walkDF[walkDF['pIDNum'].isin(keepRaters)]
tMax = np.ceil(walkDF['Event'].max() / 10) * 10
print(tMax)
# Dicts for summary data
# nRaters = {}
# nEvents = {}

# Raster & KDE

# rasterEvents = []
# for ii in walkDF['pID'].unique():
#     nRaters = len(walkDF['pID'].unique())
#     pWalkDF = walkDF[walkDF['pID'] == ii]
#     rasterEvents.append(pWalkDF['Event'].values)
# axA.eventplot(rasterEvents, linelengths = 0.75, linewidths = 2, lineoffsets = 1, colors = sns.color_palette('crest', len(rasterEvents)))
hist = sns.histplot(data = walkDF, x = 'Event', binrange = [0,tMax], 
                    binwidth = 4, color = histColor,stat = 'density', ax =axA,
                    kde = True, kde_kws= {'bw_adjust': 0.05})
# hist1 = sns.histplot(data = walkDF, x = 'Event', binrange = [0,tMax], 
#                     binwidth = 4, color = histColor,stat = 'density', ax =axB,
#                     kde = True, kde_kws= {'bw_adjust': 0.05})
# hist1 = sns.kdeplot(data = walkDF, x = 'Event',bw_adjust= 0.05, ax =axB)

# Export raw KDE vals
KDE_X, KDE_Y = hist.get_lines()[0].get_data()
KDE_DF = pd.DataFrame({'X': KDE_X, 'Y': KDE_Y})
save_str = pID + '_' + str(walk) + '_KDE.csv'
KDE_DF.to_csv(os.path.join(savepath, 'KDEs', save_str))

# KDE_X, KDE_Y = hist1.get_lines()[0].get_data()
# KDE_DF = pd.DataFrame({'X': KDE_X, 'Y': KDE_Y})
# save_str = pID + '_' + str(walk) + '_KDE1.csv'
# KDE_DF.to_csv(os.path.join(savepath, 'KDEs', save_str))

In [ ]:
# # Summary for all patients:
# for i in ESTable['RWN'].unique():
#     RWNstr = 'RWN' + str(i)
#     genRasterHist(RWNstr, save = True)

In [ ]:
# # Run for one video
# genRasterHist('RWN5', walk = 8, save = True)

# Run for all videos (no summary)
for i in ESTable['RWN'].unique():
  RWNTable = ESTable[ESTable['RWN'] == i]
  for ii in RWNTable['Walk'].unique():
    genRasterHist(('RWN' + str(i)), ii, save = True)